In [1]:
import os
from pathlib import Path
ROOT = "../" # Base directory relative to this notebook's location. Adjust if the notebook is moved
CSV_DIR = os.path.join(ROOT, "csv", "historical-data")

import pandas as pd
from IPython.display import clear_output
import time

import sys
sys.path.append(ROOT)
import src.weather_stats as ws

### Updating stats

In [2]:
rows = []

processed = 0

for filename in os.listdir(CSV_DIR):

    file_path = os.path.join(CSV_DIR, filename)

    # Ignore subfolders
    if not os.path.isfile(file_path):
        continue

    try:
        # Force indicativo as text
        df = pd.read_csv(file_path, dtype={"indicativo": str})

        clear_output(wait=True)
        print(f"Processing {filename}... ({processed + 1}/{len(os.listdir(CSV_DIR))})")
        time.sleep(0.01)

        rows.append(
            {
                "indicativo": str(df["indicativo"].iloc[0]) if "indicativo" in df.columns else None,
                "nombre": df["nombre"].iloc[0] if "nombre" in df.columns else None,
                "altitud": df["altitud"].iloc[0] if "altitud" in df.columns else None,
                "provincia": df["provincia"].iloc[0] if "provincia" in df.columns else None,
                "num_records": len(df),
                "avg_tmin": ws.safe_stat(df, "tmin", lambda s: s.mean()),
                "avg_tmax": ws.safe_stat(df, "tmax", lambda s: s.mean()),
                "avg_prec": ws.safe_stat(df, "prec", lambda s: s.mean()),
                "avg_velmedia": ws.safe_stat(df, "velmedia", lambda s: s.mean()),
                "std_tmed": ws.safe_stat(df, "tmed", lambda s: s.std()),
                "longest_gap_days": ws.calculate_longest_gap_days(df),
            }
        )

        processed += 1

    except Exception as e:
        print(f"Error processing {filename}: {e}")

stations_df = pd.DataFrame(rows)

# Ensure consistent type before sorting
stations_df["indicativo"] = stations_df["indicativo"].astype(str)

stations_df = (
    stations_df
    .sort_values("indicativo")
    .reset_index(drop=True)
)

print(f"Processed {len(stations_df)} stations.")
display(stations_df.head())

Processing C939T_hist.csv... (775/778)
Processed 775 stations.


,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed,longest_gap_days
0,0009X,ALFORJA,406,TARRAGONA,1760,10.326706,21.866588,1.275278,2.997434,6.333601,19
1,0016A,REUS AEROPUERTO,71,TARRAGONA,1950,11.727940,22.801129,1.312099,3.569625,6.325506,1
2,0034X,VALLS,233,TARRAGONA,1950,10.850077,22.453155,1.128718,NaN,6.341960,1
3,0042Y,TARRAGONA,55,TARRAGONA,1934,13.167755,22.431363,1.341408,NaN,5.705496,10
4,0061X,PONTONS,632,BARCELONA,1911,8.404505,19.591200,1.534488,3.449710,6.158788,37


In [4]:
date_cutoff = "2025-12-31"  # Set to None to include all dates

rows = []
processed = 0

for filename in os.listdir(CSV_DIR):
    file_path = os.path.join(CSV_DIR, filename)

    # Ignore subfolders
    if not os.path.isfile(file_path):
        continue

    try:
        # Force indicativo as text
        df = pd.read_csv(file_path, dtype={"indicativo": str})

        # Parse fecha to datetime (coerce invalid -> NaT)
        if "fecha" in df.columns:
            df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

        # Filter by date_cutoff if provided
        if date_cutoff is not None and "fecha" in df.columns:
            cutoff_dt = pd.to_datetime(date_cutoff)
            df = df[df["fecha"].notna() & (df["fecha"] <= cutoff_dt)]

        # Skip files with no rows after filtering
        if df.empty:
            clear_output(wait=True)
            print(f"Skipping {filename}: no rows after date filter")
            processed += 1
            time.sleep(0.005)
            continue

        clear_output(wait=True)
        print(f"Processing {filename}... ({processed + 1}/{len(os.listdir(CSV_DIR))})")
        time.sleep(0.01)

        rows.append(
            {
                "indicativo": str(df["indicativo"].iloc[0]) if "indicativo" in df.columns else None,
                "nombre": df["nombre"].iloc[0] if "nombre" in df.columns else None,
                "altitud": df["altitud"].iloc[0] if "altitud" in df.columns else None,
                "provincia": df["provincia"].iloc[0] if "provincia" in df.columns else None,
                "num_records": len(df),
                "avg_tmin": ws.safe_stat(df, "tmin", lambda s: s.mean()),
                "avg_tmax": ws.safe_stat(df, "tmax", lambda s: s.mean()),
                "avg_prec": ws.safe_stat(df, "prec", lambda s: s.mean()),
                "avg_velmedia": ws.safe_stat(df, "velmedia", lambda s: s.mean()),
                "std_tmed": ws.safe_stat(df, "tmed", lambda s: s.std()),
                "longest_gap_days": ws.calculate_longest_gap_days(df),
            }
        )

        processed += 1

    except Exception as e:
        print(f"Error processing {filename}: {e}")

stations_df = pd.DataFrame(rows)

# Ensure consistent type before sorting
stations_df["indicativo"] = stations_df["indicativo"].astype(str)

stations_df = (
    stations_df
    .sort_values("indicativo")
    .reset_index(drop=True)
)

print(f"Processed {len(stations_df)} stations.")
display(stations_df.head())

Processing C939T_hist.csv... (775/778)
Processed 775 stations.


,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed,longest_gap_days
0,0009X,ALFORJA,406,TARRAGONA,1760,10.326706,21.866588,1.275278,2.997434,6.333601,19
1,0016A,REUS AEROPUERTO,71,TARRAGONA,1826,11.984805,23.079660,1.282667,3.531507,6.397225,1
2,0034X,VALLS,233,TARRAGONA,1826,11.130849,22.759068,1.093428,NaN,6.391482,1
3,0042Y,TARRAGONA,55,TARRAGONA,1810,13.478073,22.727741,1.315155,NaN,5.729166,10
4,0061X,PONTONS,632,BARCELONA,1787,8.673445,19.903417,1.458843,3.397972,6.185819,37


In [3]:
stations_df.to_csv(os.path.join(ROOT, "csv", "stats", "weather-station-stats.csv"), index=False)

### Loading stats

In [65]:
stations_df = pd.read_csv(os.path.join(ROOT, "csv", "stats", "weather-station-stats.csv"))

### Descriptive analysis and validations

Highest Average High Temperature

In [5]:
# Highest Average High Temperature
highest_tmax = stations_df.sort_values("avg_tmax", ascending=False).head(15)
display(
    highest_tmax.style
    .set_properties(subset=["avg_tmax"], **{"background-color": "#ff9999", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed,longest_gap_days
749,C628B,"LA ALDEA DE SAN NICOLÁS, TASARTE",318,LAS PALMAS,1816,17.410094,27.256850,0.274778,3.888711,3.872089,6
375,5361X,MONTORO,155,CORDOBA,1786,11.021026,27.128861,1.276819,1.479955,7.505314,14
386,5514Z,GRANADA BASE AÉREA,695,GRANADA,1434,11.022811,26.961899,0.742575,3.310298,7.398497,397
397,5641X,ÉCIJA,130,SEVILLA,1809,12.694325,26.941941,1.220572,2.163600,7.068828,12
407,5790Y,"SEVILLA, TABLADA",9,SEVILLA,1620,13.545730,26.834839,1.306117,1.843272,6.332577,2
400,5702X,CARMONA,50,SEVILLA,1781,12.687255,26.799712,1.398119,2.732022,6.687588,22
723,C319W,"VALLEHERMOSO, DAMA",190,STA. CRUZ DE TENERIFE,1826,16.988226,26.693264,0.323768,2.512000,2.733600,0
408,5796,MORÓN DE LA FRONTERA,87,SEVILLA,1826,12.989096,26.678740,1.390925,2.003405,6.740136,1
349,4541X,EL GRANADO,60,HUELVA,1825,12.411982,26.643673,1.174523,nan,6.436653,3
378,5402,CÓRDOBA AEROPUERTO,90,CORDOBA,1826,11.513509,26.567763,1.523633,2.615427,7.343781,2


Lowest Average High Temperature

In [7]:
# Lowest Average High Temperature
lowest_tmax = stations_df.sort_values("avg_tmax", ascending=True).head(15)
display(lowest_tmax.style.set_properties(subset=["avg_tmax"], **{"background-color": "#829cd4", "color": "black"}))

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed,longest_gap_days
624,9677,PORT AINÉ,2410,LLEIDA,1753,0.705787,6.473958,nan,5.144608,6.953958,15
659,9988B,CAP DE VAQUÈIRA,2467,LLEIDA,1803,0.477296,6.605762,nan,4.164989,6.911017,15
642,9839V,"CERLER, COGULLA",2374,HUESCA,1826,1.635706,8.095400,2.320335,3.560549,6.587766,0
389,5516D,SIERRA NEVADA 'RADIOTELESCOPIO',2856,GRANADA,1619,2.569913,8.387469,nan,5.271178,7.175531,6
72,1167G,"MIRADOR DEL CABLE, PARQUE NACIONAL PICOS DE EUROPA",1910,CANTABRIA,1255,3.246018,9.619952,2.676829,5.840412,6.319316,540
73,1167J,"CORISCAO, PARQUE NACIONAL PICOS DE EUROPA",1722,CANTABRIA,1213,3.978583,9.879500,2.593403,3.278257,6.207948,531
87,1221D,PAJARES-VALGRANDE,1480,ASTURIAS,1792,4.133259,11.332137,4.371253,2.915808,5.855110,11
640,9814I,"TORLA-ORDESA, EL CEBOLLAR",1905,HUESCA,1825,4.073136,11.334759,3.316002,2.938368,6.665364,1
543,9001S,ALTO CAMPOO,1650,CANTABRIA,1733,3.902125,11.423967,3.100059,3.167520,6.057169,30
600,9451F,"PANTICOSA, PETROSOS",1850,HUESCA,1815,4.728051,11.454475,3.635502,3.120510,6.797480,9


Highest Average Low Temperature

In [8]:
# Highest Average Low Temperature
highest_tmin = stations_df.sort_values("avg_tmin", ascending=False).head(15)

display(
    highest_tmin.style
    .set_properties(subset=["avg_tmin"], **{"background-color": "#f07d12", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed,longest_gap_days
725,C329Z,SAN SEBASTIÁN DE LA GOMERA,15,STA. CRUZ DE TENERIFE,1826,20.028445,25.371166,0.294031,3.392514,2.859158,132
766,C689E,MASPALOMAS,6,LAS PALMAS,1825,19.950000,25.719643,0.138087,3.352222,2.278375,0
773,C929I,HIERRO AEROPUERTO,32,SANTA CRUZ DE TENERIFE,1826,19.907558,24.000602,0.352747,6.527653,2.246932,0
715,C229J,PÁJARA,15,LAS PALMAS,1805,19.849165,25.492631,0.192563,3.190641,2.876039,18
762,C659M,"LAS PALMAS DE GRAN CANARIA, PL. DE LA FERIA",15,LAS PALMAS,1816,19.789906,23.921479,0.367987,2.025881,2.280899,68
750,C629Q,"MOGÁN, PUERTO RICO",10,LAS PALMAS,1825,19.707139,23.957935,0.128446,3.002849,2.278609,1
745,C619X,AGAETE,5,LAS PALMAS,1825,19.609071,24.373997,0.251078,4.837699,2.595529,1
736,C449C,STA.CRUZ DE TENERIFE,36,SANTA CRUZ DE TENERIFE,1800,19.560468,25.722871,0.517420,3.016215,2.863172,4
712,C129V,FUENCALIENTE,19,STA. CRUZ DE TENERIFE,1823,19.513594,24.930655,0.478748,6.362080,2.303628,5
774,C939T,"FRONTERA, SABINOSA",20,STA. CRUZ DE TENERIFE,1826,19.277303,23.895121,0.442939,3.192430,2.345072,1


Highest Average Low Temperature excluding Canary Islands

In [9]:
# Highest Average Low Temperature excluding Canary Islands
highest_tmin_no_canary = stations_df[~stations_df["provincia"].str.contains("TENERIFE|PALMAS", na=False)] \
    .sort_values("avg_tmin", ascending=False).head(15)
display(highest_tmin_no_canary.style.set_properties(subset=["avg_tmin"], **{"background-color": "#f07d12", "color": "black"}))

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed,longest_gap_days
460,6329X,CABO DE GATA,42,ALMERIA,1806,17.897814,22.729632,0.383378,4.935778,4.932393,46
437,6083X,MARBELLA,2,MALAGA,1811,17.380161,22.725803,1.008425,3.975635,4.207317,61
683,B569X,CAPDEPERA,57,ILLES BALEARS,1790,17.246845,23.219318,1.319921,4.303375,5.504419,13
425,6000A,MELILLA,52,MELILLA,1826,16.905440,23.568022,0.684434,3.093812,4.854149,2
420,5973,CÁDIZ,2,CADIZ,1826,16.863910,22.781763,1.257667,4.480143,4.724852,0
443,6175X,RINCÓN DE LA VICTORIA,7,MALAGA,1799,16.675103,26.041848,0.980793,nan,5.101566,84
359,5000C,CEUTA,87,CEUTA,1826,16.613925,22.687226,1.834775,2.845455,4.474887,1
466,7012C,CARTAGENA,17,MURCIA,1639,16.540024,24.206593,0.734663,1.827195,5.873407,7
438,6088X,TORREMOLINOS,85,MALAGA,1826,16.524930,24.198379,1.111173,1.617902,5.153826,32
678,B398A,"CABRERA, PARQUE NACIONAL DE CABRERA",165,BALEARES,1254,16.455556,21.622684,0.989856,5.467546,5.411760,501


Lowest Average Low Temperature

In [10]:
# Lowest Average Low Temperature
lowest_tmin = stations_df.sort_values("avg_tmin", ascending=True).head(15)
display(lowest_tmin.style.set_properties(subset=["avg_tmin"], **{"background-color": "#cacbf1", "color": "black"}))

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed,longest_gap_days
659,9988B,CAP DE VAQUÈIRA,2467,LLEIDA,1803,0.477296,6.605762,nan,4.164989,6.911017,15
624,9677,PORT AINÉ,2410,LLEIDA,1753,0.705787,6.473958,nan,5.144608,6.953958,15
642,9839V,"CERLER, COGULLA",2374,HUESCA,1826,1.635706,8.095400,2.320335,3.560549,6.587766,0
264,3319D,PUERTO DEL PICO,1285,AVILA,1826,1.830782,17.114813,4.385549,2.443064,6.135031,4
618,9590D,CAP DE REC,1940,LLEIDA,1778,2.411732,11.719289,nan,1.525243,6.377491,33
389,5516D,SIERRA NEVADA 'RADIOTELESCOPIO',2856,GRANADA,1619,2.569913,8.387469,nan,5.271178,7.175531,6
203,2630X,PUERTO DE SAN ISIDRO,1510,LEON,1554,2.639327,12.534223,2.777612,3.859931,5.953648,168
212,2766E,"SANABRIA, ROBLEDA-CERVANTES",933,ZAMORA,1825,2.912301,18.394673,2.488117,0.959517,5.940367,1
72,1167G,"MIRADOR DEL CABLE, PARQUE NACIONAL PICOS DE EUROPA",1910,CANTABRIA,1255,3.246018,9.619952,2.676829,5.840412,6.319316,540
623,9657X,ESTERRI D'ÀNEU,952,LLEIDA,1798,3.416561,17.755532,1.604497,1.344563,6.375644,93


Highest Average Precipitation

In [11]:
# Highest Average Precipitation
highest_prec = stations_df.sort_values("avg_prec", ascending=False).head(15)
display(
    highest_prec.style
    .set_properties(subset=["avg_prec"], **{"background-color": "#a1cfff", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed,longest_gap_days
123,1476R,"ROIS, CASAS DO PORTO",210,A CORUÑA,1813,10.322661,19.432201,8.712900,nan,4.745783,8
113,1406X,MAZARICOS,340,A CORUÑA,1820,9.667365,17.843574,8.499669,nan,4.560922,6
126,1489A,A LAMA,395,PONTEVEDRA,1819,8.628240,18.999117,7.279192,2.246454,5.205278,7
143,1696O,BEARIZ,610,OURENSE,1821,5.379392,19.700553,6.456992,nan,5.410703,4
120,1468X,A ESTRADA,269,PONTEVEDRA,1820,9.412438,19.877545,6.323017,nan,5.124544,1
111,1399,VIMIANZO,287,A CORUÑA,1823,9.590819,18.048268,6.099059,nan,4.268093,3
127,1495,VIGO AEROPUERTO,255,PONTEVEDRA,1826,10.553495,19.617547,5.924851,3.479031,4.983942,2
39,1021X,"ERRENTERIA, AÑARBE",165,GIPUZKOA,1816,10.298567,20.155317,5.584086,1.518357,5.424791,6
147,1719,A CAÑIZA,560,PONTEVEDRA,1821,9.000440,18.579307,5.528339,nan,5.256254,4
105,1363X,AS PONTES,343,A CORUÑA,1810,8.532431,18.323407,5.443111,nan,4.854236,25


Lowest average precipitation

In [12]:
# Lowest Avg Precipitation
lowest_prec = (stations_df.dropna(subset=["avg_prec"]).sort_values("avg_prec", ascending=True).head(15))

display(
    lowest_prec.style
    .set_properties(subset=["avg_prec"], **{"background-color": "#e2d40e", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed,longest_gap_days
750,C629Q,"MOGÁN, PUERTO RICO",10,LAS PALMAS,1825,19.707139,23.957935,0.128446,3.002849,2.278609,1
766,C689E,MASPALOMAS,6,LAS PALMAS,1825,19.950000,25.719643,0.138087,3.352222,2.278375,0
716,C239N,"TUINEJE,PUERTO GRAN TARAJAL",1,LAS PALMAS,1826,18.423410,25.496765,0.146733,4.028258,3.317695,1
706,C019V,YAIZA PLAYA BLANCA,6,LAS PALMAS,1816,18.546218,24.369961,0.155592,4.131553,2.774599,10
751,C629X,"MOGÁN, PUERTO",10,LAS PALMAS,1816,18.681990,25.540300,0.167813,3.053194,2.586056,4
753,C639M,"MASPALOMAS, C. INSULAR TURISMO",45,LAS PALMAS,1820,19.093161,25.962548,0.176536,2.731648,2.952525,4
718,C249I,FUERTEVENTURA AEROPUERTO,25,LAS PALMAS,1826,18.584705,24.736665,0.177288,6.141529,2.808639,3
715,C229J,PÁJARA,15,LAS PALMAS,1805,19.849165,25.492631,0.192563,3.190641,2.876039,18
719,C258K,LA OLIVA,217,LAS PALMAS,1826,16.345884,24.041877,0.210653,3.946331,3.051647,1
717,C248E,ANTIGUA,252,LAS PALMAS,1825,15.945304,24.291105,0.212534,5.451952,3.254683,7


Lowest average precipitation excluding Canary Islands

In [13]:
# Lowest average precipitation excluding Canary Islands
lowest_prec_no_canary = (
    stations_df[~stations_df["provincia"].str.contains("TENERIFE|PALMAS", na=False)]
    .dropna(subset=["avg_prec"])
    .sort_values("avg_prec", ascending=True)
    .head(15)
)
display(
    lowest_prec_no_canary.style.set_properties(subset=["avg_prec"], **{"background-color": "#e2d40e", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed,longest_gap_days
454,6291B,EL EJIDO,98,ALMERIA,1826,16.186804,25.871437,0.264622,nan,5.399789,30
455,6293X,ROQUETAS DE MAR,3,ALMERIA,1624,14.464521,23.261232,0.316401,4.211813,5.050131,265
446,6205X,TORROX,3,MALAGA,1639,15.424954,23.309427,0.380775,3.190161,4.478553,608
460,6329X,CABO DE GATA,42,ALMERIA,1806,17.897814,22.729632,0.383378,4.935778,4.932393,46
459,6325O,ALMERÍA AEROPUERTO,21,ALMERIA,1826,16.051068,24.536110,0.507403,4.537946,5.607652,1
361,5047E,BAZA,785,GRANADA,1726,8.168757,24.264302,0.541782,3.007587,7.521833,29
462,6364X,ALBOX,508,ALMERIA,1822,13.284344,25.558214,0.577486,3.416465,6.610087,5
461,6340X,GARRUCHA,28,ALMERIA,1824,15.726898,23.039109,0.618781,3.978991,5.340339,2
451,6268Y,"MOTRIL, PUERTO",3,GRANADA,1826,15.688650,23.053303,0.657748,2.519504,4.621726,19
465,7007Y,MAZARRÓN,66,MURCIA,1826,15.602912,25.209116,0.669989,3.210159,5.705251,3


Highest average wind speed

In [14]:
# Highest average wind speed
highest_wind = stations_df.sort_values("avg_velmedia", ascending=False).head(15)
display(
    highest_wind.style
    .set_properties(subset=["avg_velmedia"], **{"background-color": "#ff99f7", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed,longest_gap_days
104,1351,ESTACA DE BARES,90,A CORUÑA,1821,12.760933,17.046768,2.190676,8.332233,3.389095,32
757,C649I,GRAN CANARIA AEROPUERTO,24,LAS PALMAS,1826,18.988259,25.260068,0.358204,8.185989,2.748513,11
720,C314Z,"VALLEHERMOSO, ALTO IGUALERO",1474,STA. CRUZ DE TENERIFE,1803,10.680989,18.308551,1.279887,6.867056,6.091209,10
731,C430E,IZAÑA,2369,SANTA CRUZ DE TENERIFE,1826,7.337733,15.291073,0.609740,6.800227,5.924358,0
112,1400,FISTERRA,230,A CORUÑA,1825,12.039250,17.204410,2.930643,6.722864,3.678081,2
110,1393,CABO VILÁN,50,A CORUÑA,1797,12.156906,17.196119,3.353004,6.709631,3.150132,17
773,C929I,HIERRO AEROPUERTO,32,SANTA CRUZ DE TENERIFE,1826,19.907558,24.000602,0.352747,6.527653,2.246932,0
712,C129V,FUENCALIENTE,19,STA. CRUZ DE TENERIFE,1823,19.513594,24.930655,0.478748,6.362080,2.303628,5
730,C429I,TENERIFE SUR AEROPUERTO,64,SANTA CRUZ DE TENERIFE,1826,18.375247,26.218695,0.285253,6.257915,2.996161,1
718,C249I,FUERTEVENTURA AEROPUERTO,25,LAS PALMAS,1826,18.584705,24.736665,0.177288,6.141529,2.808639,3


Highest average wind speed excluding Canary Islands

In [15]:
# Highest average wind speed excluding Canary Islands
highest_wind_no_canary = stations_df[~stations_df["provincia"].str.contains("TENERIFE|PALMAS", na=False)] \
    .sort_values("avg_velmedia", ascending=False).head(15)
display(
    highest_wind_no_canary.style
    .set_properties(subset=["avg_velmedia"], **{"background-color": "#ff99f7", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed,longest_gap_days
104,1351,ESTACA DE BARES,90,A CORUÑA,1821,12.760933,17.046768,2.190676,8.332233,3.389095,32
112,1400,FISTERRA,230,A CORUÑA,1825,12.039250,17.204410,2.930643,6.722864,3.678081,2
110,1393,CABO VILÁN,50,A CORUÑA,1797,12.156906,17.196119,3.353004,6.709631,3.150132,17
426,6001,TARIFA,32,CADIZ,1818,15.385077,21.007324,1.720617,6.103239,4.233314,3
72,1167G,"MIRADOR DEL CABLE, PARQUE NACIONAL PICOS DE EUROPA",1910,CANTABRIA,1255,3.246018,9.619952,2.676829,5.840412,6.319316,540
678,B398A,"CABRERA, PARQUE NACIONAL DE CABRERA",165,BALEARES,1254,16.455556,21.622684,0.989856,5.467546,5.411760,501
190,2491C,"LA COVATILLA, ESTACIÓN DE ESQUÍ",1960,SALAMANCA,1740,4.371751,11.856071,3.917062,5.311436,6.771521,95
389,5516D,SIERRA NEVADA 'RADIOTELESCOPIO',2856,GRANADA,1619,2.569913,8.387469,nan,5.271178,7.175531,6
610,9550C,"ANDORRA, HORCALLANA",762,TERUEL,1810,9.698606,20.056247,0.871148,5.233778,7.254585,5
624,9677,PORT AINÉ,2410,LLEIDA,1753,0.705787,6.473958,nan,5.144608,6.953958,15


Lowest average wind speed

In [16]:
# Lowest average wind speed
lowest_wind = (
    stations_df.dropna(subset=["avg_velmedia"])
    .sort_values("avg_velmedia", ascending=True)
    .head(15)
)
display(
    lowest_wind.style
    .set_properties(subset=["avg_velmedia"], **{"background-color": "#99ff99", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed,longest_gap_days
428,6040X,CORTES DE LA FRONTERA,315,MALAGA,1787,12.177083,23.979899,2.236824,0.713185,6.050135,48
24,0360X,LES PLANES D'HOSTOLES,337,GIRONA,1813,6.973019,21.776343,2.036410,0.745382,6.607291,9
144,1700X,O CARBALLIÑO,400,OURENSE,1817,7.652925,20.787298,3.331291,0.774359,5.930688,7
667,B103B,ANDRATX - SANT ELM,52,BALEARES,1802,13.393746,23.124268,1.019441,0.834097,6.011311,12
601,9453X,"BIESCAS, EMBALSE DE BÚBAL",1100,HUESCA,1826,4.896487,16.895283,4.024326,0.865625,6.546745,1
388,5515X,GRANADA-CARTUJA,775,GRANADA,1815,11.257951,25.300784,0.750990,0.900056,7.577030,17
274,3423I,MADRIGAL DE LA VERA,464,CACERES,1826,11.453524,23.337775,3.457644,0.908026,7.471196,4
664,B013X,"ESCORCA, LLUC",490,ILLES BALEARS,1821,10.004972,21.553536,2.675228,0.927996,6.449458,4
643,9843A,SEIRA,825,HUESCA,1826,6.073644,19.772204,2.908055,0.949069,7.019642,1
212,2766E,"SANABRIA, ROBLEDA-CERVANTES",933,ZAMORA,1825,2.912301,18.394673,2.488117,0.959517,5.940367,1


Highest Temperature Variability (std of tmed)

In [17]:
# Highest Temperature Variability (std of tmed)
highest_variability = stations_df.sort_values("std_tmed", ascending=False).head(15)
display(
    highest_variability.style
    .set_properties(subset=["std_tmed"], **{"background-color": "#c80dda", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed,longest_gap_days
300,4064Y,ALCAZAR DE SAN JUAN,640,CIUDAD REAL,1812,10.916203,23.417974,0.857508,1.603485,8.413925,16
622,9650X,ARTESA DE SEGRE,400,LLEIDA,1826,7.937466,21.791333,1.037783,nan,8.170128,1
239,3085Y,PASTRANA,920,GUADALAJARA,1699,8.270062,21.247823,1.419388,nan,8.164714,91
314,4147X,VALDEPEÑAS,700,CIUDAD REAL,1819,10.284840,23.157245,0.822838,1.784040,8.140728,3
312,4121,CIUDAD REAL,626,CIUDAD REAL,1826,10.855099,23.338740,1.055889,2.085305,8.122333,1
627,9707,LLIMIANA,515,LLEIDA,1775,7.722994,22.597097,1.431411,1.814844,8.107595,38
311,4116I,ALMAGRO / FAMET,626,CIUDAD REAL,1826,8.984375,23.404331,0.955982,2.863570,8.107373,1
310,4103X,TOMELLOSO,662,CIUDAD REAL,1772,9.499943,23.580909,0.836280,2.725637,8.106879,18
315,4148,VISO DEL MARQUÉS,804,CIUDAD REAL,1769,9.610413,22.643294,1.160793,2.690554,8.087475,37
252,3182Y,ARGANDA DEL REY,533,MADRID,1762,9.269151,23.360493,1.063278,2.405533,8.082031,38


Lowest Temperature Variability (std of tmed)

In [18]:
# Lowest Temperature Variability (std of tmed)
lowest_variability = (
    stations_df.dropna(subset=["std_tmed"])
    .sort_values("std_tmed", ascending=True)
    .head(15)
)
display(
    lowest_variability.style.set_properties(subset=["std_tmed"], **{"background-color": "#cbb0cc", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed,longest_gap_days
773,C929I,HIERRO AEROPUERTO,32,SANTA CRUZ DE TENERIFE,1826,19.907558,24.000602,0.352747,6.527653,2.246932,0
766,C689E,MASPALOMAS,6,LAS PALMAS,1825,19.950000,25.719643,0.138087,3.352222,2.278375,0
750,C629Q,"MOGÁN, PUERTO RICO",10,LAS PALMAS,1825,19.707139,23.957935,0.128446,3.002849,2.278609,1
762,C659M,"LAS PALMAS DE GRAN CANARIA, PL. DE LA FERIA",15,LAS PALMAS,1816,19.789906,23.921479,0.367987,2.025881,2.280899,68
761,C659H,"LAS PALMAS DE GRAN CANARIA, SAN CRISTOBAL",55,LAS PALMAS,1800,18.614286,23.935569,0.338149,3.957111,2.281627,329
712,C129V,FUENCALIENTE,19,STA. CRUZ DE TENERIFE,1823,19.513594,24.930655,0.478748,6.362080,2.303628,5
774,C939T,"FRONTERA, SABINOSA",20,STA. CRUZ DE TENERIFE,1826,19.277303,23.895121,0.442939,3.192430,2.345072,1
713,C139E,LA PALMA AEROPUERTO,33,SANTA CRUZ DE TENERIFE,1826,18.955044,23.695724,0.755178,5.313216,2.361413,1
740,C459Z,PUERTO DE LA CRUZ,25,STA. CRUZ DE TENERIFE,1821,18.695815,24.898403,0.660220,2.660780,2.413779,5
765,C669B,ARUCAS,86,LAS PALMAS,1823,18.106019,23.735560,0.382900,2.501756,2.431425,2


Lowest temperature variability excluding Canary Islands

In [19]:
# Lowest temperature variability excluding Canary Islands
lowest_variability_no_canary = (
    stations_df[~stations_df["provincia"].str.contains("TENERIFE|PALMAS", na=False)]
    .dropna(subset=["std_tmed"])
    .sort_values("std_tmed", ascending=True)
    .head(15)
)
display(
    lowest_variability_no_canary.style.set_properties(subset=["std_tmed"], **{"background-color": "#cbb0cc", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed,longest_gap_days
110,1393,CABO VILÁN,50,A CORUÑA,1797,12.156906,17.196119,3.353004,6.709631,3.150132,17
104,1351,ESTACA DE BARES,90,A CORUÑA,1821,12.760933,17.046768,2.190676,8.332233,3.389095,32
112,1400,FISTERRA,230,A CORUÑA,1825,12.039250,17.204410,2.930643,6.722864,3.678081,2
106,1387,A CORUÑA,57,A CORUÑA,1826,12.843209,18.963198,3.193184,3.564863,3.776912,0
103,1347T,BURELA,80,LUGO,1824,12.321197,17.970291,2.526596,nan,3.810380,1
107,1387D,A CORUÑA BENS,132,A CORUÑA,1797,11.892857,17.949745,3.117536,4.212927,3.829470,18
95,1283U,CABO BUSTO,60,ASTURIAS,1745,11.917197,18.044239,1.959918,4.663930,3.883895,13
85,1210X,CABO PEÑAS,100,ASTURIAS,1747,12.807573,17.745200,2.398268,3.929436,3.892409,28
79,1183X,LLANES,10,ASTURIAS,1794,12.225280,18.763031,2.884422,2.636034,4.072188,12
101,1342X,RIBADEO,43,LUGO,1800,11.293785,18.755350,2.463434,3.127308,4.074734,14


Highest daily temperature range (avg_tmax - avg_tmin)

In [20]:
# Highest daily temperature range (avg_tmax - avg_tmin)
stations_df["avg_temp_range"] = stations_df["avg_tmax"] - stations_df["avg_tmin"]
highest_temp_range = stations_df.sort_values("avg_temp_range", ascending=False).head(15)
display(
    highest_temp_range.style
    .set_properties(subset=["avg_temp_range"], **{"background-color": "#e9430c", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed,longest_gap_days,avg_temp_range
168,2192C,CUÉLLAR,795,SEGOVIA,1824,3.459638,21.095774,1.324822,2.527991,6.894900,3,17.636136
390,5530E,GRANADA AEROPUERTO,560,GRANADA,1826,8.868257,25.559320,0.885430,2.612864,7.623299,1,16.691064
398,5654X,LA PUEBLA DE LOS INFANTES,200,SEVILLA,1813,9.204892,25.829366,1.917614,1.857654,7.073461,6,16.624474
617,9590,MARTINET,1038,LLEIDA,1826,3.668586,20.206524,1.653497,1.611671,7.113665,1,16.537939
516,8245Y,MIRA,815,CUENCA,1396,5.551400,21.815506,1.070173,nan,7.004783,2,16.264106
527,8381X,ADEMUZ,705,VALENCIA,1826,6.462239,22.685840,1.017415,nan,7.252828,3,16.223600
375,5361X,MONTORO,155,CORDOBA,1786,11.021026,27.128861,1.276819,1.479955,7.505314,14,16.107835
361,5047E,BAZA,785,GRANADA,1726,8.168757,24.264302,0.541782,3.007587,7.521833,29,16.095545
166,2172Y,SARDÓN DE DUERO,725,VALLADOLID,1773,5.235066,21.324177,1.141919,2.222680,7.083700,12,16.089111
348,4527X,AROCHE,267,HUELVA,1822,8.893626,24.889396,1.622895,2.045993,6.465174,2,15.995769


Lowest daily temperature range (avg_tmax - avg_tmin)

In [21]:
# Lowest daily temperature range (avg_tmax - avg_tmin)
lowest_temp_range = (
    stations_df.dropna(subset=["avg_temp_range"])
    .sort_values("avg_temp_range", ascending=True)
    .head(15)
)
display(
    lowest_temp_range.style.set_properties(subset=["avg_temp_range"], **{"background-color": "#99e699", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed,longest_gap_days,avg_temp_range
773,C929I,HIERRO AEROPUERTO,32,SANTA CRUZ DE TENERIFE,1826,19.907558,24.000602,0.352747,6.527653,2.246932,0,4.093045
762,C659M,"LAS PALMAS DE GRAN CANARIA, PL. DE LA FERIA",15,LAS PALMAS,1816,19.789906,23.921479,0.367987,2.025881,2.280899,68,4.131573
750,C629Q,"MOGÁN, PUERTO RICO",10,LAS PALMAS,1825,19.707139,23.957935,0.128446,3.002849,2.278609,1,4.250796
104,1351,ESTACA DE BARES,90,A CORUÑA,1821,12.760933,17.046768,2.190676,8.332233,3.389095,32,4.285835
34,0433D,CABO DE CREUS,75,GIRONA,1769,15.833857,20.362965,0.870502,nan,5.641124,6,4.529108
774,C939T,"FRONTERA, SABINOSA",20,STA. CRUZ DE TENERIFE,1826,19.277303,23.895121,0.442939,3.192430,2.345072,1,4.617818
713,C139E,LA PALMA AEROPUERTO,33,SANTA CRUZ DE TENERIFE,1826,18.955044,23.695724,0.755178,5.313216,2.361413,1,4.740680
745,C619X,AGAETE,5,LAS PALMAS,1825,19.609071,24.373997,0.251078,4.837699,2.595529,1,4.764926
460,6329X,CABO DE GATA,42,ALMERIA,1806,17.897814,22.729632,0.383378,4.935778,4.932393,46,4.831818
772,C928I,VALVERDE,670,STA. CRUZ DE TENERIFE,1826,14.316922,19.179452,0.870475,5.321742,3.186581,1,4.862530


Lowest daily temperature range (avg_tmax - avg_tmin) excluding Canary Islands

In [22]:
# Lowest daily temperature range (avg_tmax - avg_tmin) excluding Canary Islands
lowest_temp_range_no_canary = (
    stations_df[~stations_df["provincia"].str.contains("TENERIFE|PALMAS", na=False)]
    .dropna(subset=["avg_temp_range"])
    .sort_values("avg_temp_range", ascending=True)
    .head(15)
)
display(
    lowest_temp_range_no_canary.style.set_properties(subset=["avg_temp_range"], **{"background-color": "#99e699", "color": "black"})
)

,indicativo,nombre,altitud,provincia,num_records,avg_tmin,avg_tmax,avg_prec,avg_velmedia,std_tmed,longest_gap_days,avg_temp_range
104,1351,ESTACA DE BARES,90,A CORUÑA,1821,12.760933,17.046768,2.190676,8.332233,3.389095,32,4.285835
34,0433D,CABO DE CREUS,75,GIRONA,1769,15.833857,20.362965,0.870502,nan,5.641124,6,4.529108
460,6329X,CABO DE GATA,42,ALMERIA,1806,17.897814,22.729632,0.383378,4.935778,4.932393,46,4.831818
85,1210X,CABO PEÑAS,100,ASTURIAS,1747,12.807573,17.745200,2.398268,3.929436,3.892409,28,4.937627
110,1393,CABO VILÁN,50,A CORUÑA,1797,12.156906,17.196119,3.353004,6.709631,3.150132,17,5.039212
112,1400,FISTERRA,230,A CORUÑA,1825,12.039250,17.204410,2.930643,6.722864,3.678081,2,5.165160
678,B398A,"CABRERA, PARQUE NACIONAL DE CABRERA",165,BALEARES,1254,16.455556,21.622684,0.989856,5.467546,5.411760,501,5.167128
53,1057B,MATXITXAKO,93,BIZKAIA,1826,13.196706,18.455835,3.352539,5.114786,4.522474,34,5.259129
65,1111X,SANTANDER,51,CANTABRIA,1826,12.977607,18.314874,3.401656,4.585151,4.249904,2,5.337267
437,6083X,MARBELLA,2,MALAGA,1811,17.380161,22.725803,1.008425,3.975635,4.207317,61,5.345642
